# 01. Olist Data Audit

## Objective

Validate dataset grain, keys, customer identity, order lifecycle fields, and temporal coverage before constructing the 90-day repeat-purchase cohort.

In [1]:
# 패키지
import pandas as pd

In [2]:
# 데이터 로딩
customers_df = pd.read_csv(
    "../data/raw/olist_customers_dataset.csv"
)

orders_df = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv"
)

order_items_df = pd.read_csv(
    "../data/raw/olist_order_items_dataset.csv"
)

payments_df = pd.read_csv(
    "../data/raw/olist_order_payments_dataset.csv"
)

products_df = pd.read_csv(
    "../data/raw/olist_products_dataset.csv"
)

category_translation_df = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

In [3]:
# 데이터 크기 확인
print("customers")
print(customers_df.shape)

print()

print("orders")
print(orders_df.shape)

print()

print("order_items")
print(order_items_df.shape)

print()

print("payments")
print(payments_df.shape)

print()

print("products")
print(products_df.shape)

print()

print("category_translation")
print(category_translation_df.shape)

customers
(99441, 5)

orders
(99441, 8)

order_items
(112650, 7)

payments
(103886, 5)

products
(32951, 9)

category_translation
(71, 2)


In [4]:
# 칼럼 확인
print(customers_df.columns.tolist())

print()
print(orders_df.columns.tolist())

print()
print(order_items_df.columns.tolist())

print()
print(payments_df.columns.tolist())

print()
print(products_df.columns.tolist())

['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [5]:
# 고객 식별자 검증
print(
    "customer_id:",
    customers_df["customer_id"].nunique()
)

print(
    "customer_unique_id:",
    customers_df["customer_unique_id"].nunique()
)

print(
    "customer_id duplicates:",
    customers_df["customer_id"]
    .duplicated()
    .sum()
)

print(
    "customer_unique_id duplicates:",
    customers_df["customer_unique_id"]
    .duplicated()
    .sum()
)

customer_id: 99441
customer_unique_id: 96096
customer_id duplicates: 0
customer_unique_id duplicates: 3345


### Finding — Customer Identity

customer_id is unique across all 99,441 customer records, while only 96,096 unique customer_unique_id values exist.

Therefore customer_unique_id will be used as the customer-level analysis key for repeat-purchase tracking.

In [6]:
# 주문 key 검증
print(
    "orders order_id duplicate:",
    orders_df["order_id"]
    .duplicated()
    .sum()
)

print(
    "order_items duplicated order_id:",
    order_items_df["order_id"]
    .duplicated()
    .sum()
)

print(
    "payments duplicated order_id:",
    payments_df["order_id"]
    .duplicated()
    .sum()
)

orders order_id duplicate: 0
order_items duplicated order_id: 13984
payments duplicated order_id: 4446


### Finding — One-to-Many Relationships

order_id is unique in orders but repeats in both order_items and payments.

These tables must be aggregated to order level before being joined together to avoid row multiplication.

In [7]:
# 주문 상태 Audit
print(
    orders_df["order_status"]
    .value_counts(
        dropna=False
    )
)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [8]:
# 날짜 Audit
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders_df[column] = pd.to_datetime(
        orders_df[column]
    )

print(
    orders_df[
        date_columns
    ]
    .isna()
    .sum()
)

print()

# 기간
print(
    orders_df["order_purchase_timestamp"].min()
)

print(
    orders_df["order_purchase_timestamp"].max()
)

print()

print(
    orders_df["order_approved_at"].min()
)

print(
    orders_df["order_approved_at"].max()
)

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

2016-09-04 21:15:19
2018-10-17 17:30:18

2016-09-15 12:16:38
2018-09-03 17:40:06


In [9]:
# 승인 시점 결측 확인
approved_missing_df = (
    orders_df[
        orders_df["order_approved_at"].isna()
    ]
)

print(
    approved_missing_df[
        "order_status"
    ]
    .value_counts(
        dropna=False
    )
)

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


### Finding — Approval Timestamp Quality

160 orders do not have order_approved_at.
Most are canceled, but 14 delivered orders are also affected.

The project will therefore not finalize the prediction
snapshot until these cases are reviewed.

In [10]:
# Composite Key 검증
print(
    "order_items composite key duplicates:",
    order_items_df[
        ["order_id", "order_item_id"]
    ]
    .duplicated()
    .sum()
)

print(
    "payments composite key duplicates:",
    payments_df[
        ["order_id", "payment_sequential"]
    ]
    .duplicated()
    .sum()
)

print(
    "products product_id duplicates:",
    products_df["product_id"]
    .duplicated()
    .sum()
)

print(
    "category duplicates:",
    category_translation_df[
        "product_category_name"
    ]
    .duplicated()
    .sum()
)

order_items composite key duplicates: 0
payments composite key duplicates: 0
products product_id duplicates: 0
category duplicates: 0


In [11]:
# 관계 무결성
print(
    "orders without customer:",
    (~orders_df["customer_id"].isin(
        customers_df["customer_id"]
    ))
    .sum()
)

print(
    "order_items without order:",
    (~order_items_df["order_id"].isin(
        orders_df["order_id"]
    ))
    .sum()
)

print(
    "payments without order:",
    (~payments_df["order_id"].isin(
        orders_df["order_id"]
    ))
    .sum()
)

print(
    "order_items without product:",
    (~order_items_df["product_id"].isin(
        products_df["product_id"]
    ))
    .sum()
)

orders without customer: 0
order_items without order: 0
payments without order: 0
order_items without product: 0


In [12]:
# 날짜 범위 이상한 부분 확인
orders_df[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at"
    ]
].sort_values(
    "order_purchase_timestamp",
    ascending=False
).head(10)

,order_id,order_status,order_purchase_timestamp,order_approved_at
60938,10a045cdf6a5650c21e9cfeb60384c16,canceled,2018-10-17 17:30:18,NaT
68373,b059ee4de278302d550a3035c4cdb740,canceled,2018-10-16 20:16:02,NaT
31891,a2ac6dad85cf8af5b0afb510a240fe8c,canceled,2018-10-03 18:55:29,NaT
88500,616fa7d4871b87832197b2a137a115d2,canceled,2018-10-01 15:30:09,NaT
50387,392ed9afd714e3c74767d0c4d3e3f477,canceled,2018-09-29 09:13:03,NaT
37003,869997fbe01f39d184956b5c6bccfdbe,canceled,2018-09-26 08:40:15,NaT
33979,5aac76cf7b07dd06fa4d50bf461d2f40,canceled,2018-09-25 11:59:18,NaT
1801,ed3efbd3a87bea76c2812c66a0b32219,canceled,2018-09-20 13:54:16,NaT
16366,bd35b677fd239386e9861d11ae98ab56,canceled,2018-09-17 17:21:16,NaT
5149,ea844c92cf978ea23321fa7fe5871761,canceled,2018-09-13 09:56:12,NaT
